In [1]:
import pandas as pd

In [2]:
# =====================================================
# Load Dataset & Basic Preprocessing
# =====================================================

df = pd.read_csv("../Raw Data/products.csv")
df['launch_date'] = pd.to_datetime(df['launch_date'])
df.head()

,product_id,product_name,category,sub_category,seller_id,cost_price,selling_price,launch_date
0,PROD000001,Advanced Self-Help 337,Books,Self-Help,SELR000473,923.95,1100.46,2024-10-18
1,PROD000002,Superior Kids Wear 443,Fashion,Kids Wear,SELR000744,2015.54,2774.35,2025-03-28
2,PROD000003,Basic Cycling 243,Sports & Fitness,Cycling,SELR001084,1182.82,1575.92,2023-03-09
3,PROD000004,Superior Beverages 987,Grocery,Beverages,SELR000978,45.18,51.07,2023-07-31
4,PROD000005,Ultra Haircare 256,Beauty & Personal Care,Haircare,SELR000669,763.01,990.10,2024-01-15


In [3]:
df.sample(10)

,product_id,product_name,category,sub_category,seller_id,cost_price,selling_price,launch_date
3116,PROD003117,Compact Outdoor Gear 470,Sports & Fitness,Outdoor Gear,SELR000735,3357.62,4151.29,2023-11-27
1315,PROD001316,Deluxe Cookware 959,Home & Kitchen,Cookware,SELR001108,1033.26,1424.19,2024-02-06
3052,PROD003053,Smart Bag 387,Fashion,Bags,SELR000076,879.25,1433.26,2023-09-10
1077,PROD001078,Smart Cookware 414,Home & Kitchen,Cookware,SELR000173,951.61,1282.19,2023-06-25
3170,PROD003171,Essential Footwear 993,Fashion,Footwear,SELR000634,1311.34,2059.14,2023-12-26
1581,PROD001582,Ultra Accessories 873,Electronics,Accessories,SELR000859,899.06,1069.26,2023-12-14
989,PROD000990,Basic Bag 364,Fashion,Bags,SELR001026,2483.32,3799.77,2023-12-20
168,PROD000169,Essential Women's Clothing 302,Fashion,Women's Clothing,SELR000979,448.62,664.58,2025-01-29
1178,PROD001179,Premium Children's Book 637,Books,Children's Books,SELR000261,280.02,333.07,2024-01-06
1184,PROD001185,Everyday Storage 443,Home & Kitchen,Storage,SELR000968,744.32,997.27,2023-10-13


In [4]:
print("Total Rows In The Dataset:",df.shape[0])
print("Total Columns In The Dataset:",df.shape[1])

Total Rows In The Dataset: 3500
Total Columns In The Dataset: 8


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   product_id     3500 non-null   object        
 1   product_name   3500 non-null   object        
 2   category       3486 non-null   object        
 3   sub_category   3465 non-null   object        
 4   seller_id      3500 non-null   object        
 5   cost_price     3500 non-null   float64       
 6   selling_price  3500 non-null   float64       
 7   launch_date    3500 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 218.9+ KB


In [6]:
df.describe()

,cost_price,selling_price,launch_date
count,3500.000000,3500.000000,3500
mean,3625.119291,4587.284594,2023-11-29 09:49:09.942856960
min,30.200000,33.200000,2022-11-27 00:00:00
25%,427.980000,572.670000,2023-05-04 00:00:00
50%,1097.680000,1452.290000,2023-09-28 00:00:00
75%,2903.027500,4040.155000,2024-02-29 00:00:00
max,63907.010000,77934.410000,2025-10-21 00:00:00
std,7836.885617,9409.035133,NaN


In [7]:
# String Cleaning:
df['product_name'] = (df["product_name"]
                .str.strip()
                .str.lower()
                .str.title()
)

df['category'] = (df["category"]
                .str.strip()
                .str.lower()
                .str.title()
)

df['sub_category'] = (df["sub_category"]
                .str.strip()
                .str.lower()
                .str.title()
)

In [8]:
print("Total Missing Values In Columns:")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    
result = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_pct
})
    
print(result[result["missing_count"] > 0])

Total Missing Values In Columns:
              missing_count  missing_percent
category                 14              0.4
sub_category             35              1.0


In [9]:
df[df['category'].isna() | df["sub_category"].isna()][['product_id','product_name', 'category', 'sub_category']]

,product_id,product_name,category,sub_category
31,PROD000032,Advanced Toys 535,NaN,Toys
66,PROD000067,Pro Footwear 181,Fashion,NaN
152,PROD000153,Essential Bags 262,Fashion,NaN
191,PROD000192,Trendy Fragrances 407,NaN,Fragrances
198,PROD000199,Classic Storage 133,Home & Kitchen,NaN
236,PROD000237,Comfort Children'S Books 771,Books,NaN
252,PROD000253,Elite Baby Care 595,Toys & Baby Products,NaN
263,PROD000264,Signature Smart Watche 569,NaN,Smart Watches
341,PROD000342,Ultra Children'S Books 811,Books,NaN
439,PROD000440,Deluxe Skincare 415,Beauty & Personal Care,NaN


In [10]:
# Build sub_category -> category mapping from clean rows
mapping = df.dropna(subset=['category', 'sub_category']) \
            .groupby('sub_category')['category'] \
            .agg(lambda x: x.mode()[0])  # mode() in case of rare conflicts

# Check that mapping is actually clean (1:1)
check = df.dropna(subset=['category', 'sub_category']).groupby('sub_category')['category'].nunique()
print(check[check > 1])   # ideally empty output — confirms 1 sub_category = 1 category always

# Fill missing category using the mapping
df['category'] = df['category'].fillna(df['sub_category'].map(mapping))

Series([], Name: category, dtype: int64)


In [11]:
import re

# Get all known sub-categories
subcategories = (
    df["sub_category"]
    .dropna()
    .unique()
    .tolist()
)

subcategories = sorted(subcategories, key=len, reverse=True)

# Create regex pattern
pattern = "|".join(re.escape(x) for x in subcategories)

# Extract sub-category from product name
extracted_subcategory = df["product_name"].str.extract(
    f"({pattern})",
    flags=re.IGNORECASE,
    expand=False
)

# Fill missing sub-category values
df["sub_category"] = df["sub_category"].fillna(extracted_subcategory)

In [12]:
print("Total Duplicates Rows In product_id:",df.duplicated(subset=["product_id"]).sum())

Total Duplicates Rows In product_id: 0


In [13]:
print("Total Product:", df["product_name"].nunique())
print("Total Sellers:", df["seller_id"].nunique())

Total Product: 3490
Total Sellers: 1065


In [14]:
print("Total Category:", df["category"].nunique(),"\n")
print(df["category"].value_counts())

Total Category: 8 

category
Fashion                   819
Home & Kitchen            573
Electronics               461
Beauty & Personal Care    412
Toys & Baby Products      390
Sports & Fitness          329
Grocery                   262
Books                     254
Name: count, dtype: int64


In [15]:
print("Total Sub_Category:", df["sub_category"].nunique(),"\n")
print(df["sub_category"].value_counts())

Total Sub_Category: 49 

sub_category
Kids Wear                   161
Watches & Jewellery         145
Women'S Clothing            135
Bags                        128
Footwear                    125
Men'S Clothing              125
Bedding                     111
Furniture                   108
Storage                     100
Kitchen Appliances           96
Cookware                     93
Fragrances                   80
Makeup                       76
Diapers                      76
Smartphones                  74
Toys                         74
Grooming Tools               74
Accessories                  73
Cameras                      72
Baby Feeding                 69
Haircare                     68
Headphones                   66
Home Decor                   65
Skincare                     65
Baby Gear                    63
Outdoor Gear                 62
Cycling                      62
Tablets                      60
Baby Care                    59
Smart Watches                58
La

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   product_id     3500 non-null   object        
 1   product_name   3500 non-null   object        
 2   category       3500 non-null   object        
 3   sub_category   3500 non-null   object        
 4   seller_id      3500 non-null   object        
 5   cost_price     3500 non-null   float64       
 6   selling_price  3500 non-null   float64       
 7   launch_date    3500 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 218.9+ KB


In [17]:
df.sample(10)

,product_id,product_name,category,sub_category,seller_id,cost_price,selling_price,launch_date
1192,PROD001193,Compact Home Decor 160,Home & Kitchen,Home Decor,SELR000683,2324.06,2960.15,2023-12-31
915,PROD000916,Smart Packaged Food 604,Grocery,Packaged Food,SELR000827,378.81,422.51,2023-12-16
821,PROD000822,Compact Personal Hygiene 159,Beauty & Personal Care,Personal Hygiene,SELR000601,791.17,998.04,2023-05-18
613,PROD000614,Smart Staples 966,Grocery,Staples,SELR000940,614.44,717.25,2023-04-16
758,PROD000759,Signature Headphone 162,Electronics,Headphones,SELR000176,25142.13,30752.47,2023-11-22
1958,PROD001959,Basic Smartphones 494,Electronics,Smartphones,SELR000424,1157.06,1442.30,2024-02-29
2503,PROD002504,Basic Women'S Clothing 786,Fashion,Women'S Clothing,SELR000234,2076.19,3355.33,2023-03-29
2732,PROD002733,Compact Cookware 664,Home & Kitchen,Cookware,SELR000683,1265.71,1775.32,2023-10-26
3498,PROD003499,Premium Team Sports 461,Sports & Fitness,Team Sports,SELR000606,467.20,647.02,2025-05-13
2239,PROD002240,Ultra Storage 455,Home & Kitchen,Storage,SELR001053,2450.81,3405.48,2023-06-18


In [18]:
df.to_csv("../Clean Data/products.csv",index=False)

In [19]:
print(f"Memory Usage: {df.memory_usage(deep=True).sum()/1024**2:.2f} MB")
print(df.dtypes)

Memory Usage: 1.25 MB
product_id               object
product_name             object
category                 object
sub_category             object
seller_id                object
cost_price              float64
selling_price           float64
launch_date      datetime64[ns]
dtype: object
